In [1]:
import pandas as pd
import numpy as np
import datetime

print("--- FASE 1: ETL-Prozess (Bereinigung & Imputation mit Pandas) ---")

# 1. Rohdatensimulation (Forschungspraktikum ZEE)
def generiere_rohdaten():
    np.random.seed(42)
    zeitstempel = [datetime.datetime(2026, 6, 1) + datetime.timedelta(minutes=i*15) for i in range(100)]
    
    daten = {
        "zeitstempel": zeitstempel,
        "technologie": np.random.choice(["PERC", "HIT", "CIGS"], size=100),
        "einstrahlung_wm2": np.random.uniform(200, 1000, size=100),
        "leistung_kw": np.random.uniform(5, 50, size=100)
    }
    
    df = pd.DataFrame(daten)
    # Künstliche Fehlwerte einfügen für die Demonstration der Bereinigung
    df.loc[df["einstrahlung_wm2"] < 250, "leistung_kw"] = np.nan
    return df

df_rohdaten = generiere_rohdaten()
print(f"-> Rohdaten erfolgreich geladen. Fehlwerte in 'leistung_kw': {df_rohdaten['leistung_kw'].isna().sum()}")

# 2. Data Engineering: Imputation der Fehlwerte durch den Mittelwert der jeweiligen Technologie
df_bereinigt = df_rohdaten.copy()
df_bereinigt["leistung_kw"] = df_bereinigt.groupby("technologie")["leistung_kw"].transform(lambda x: x.fillna(x.mean()))

# 3. Performance-Modellierung (HIT auf exakt 82% setzen laut Lebenslauf)
df_bereinigt["performance_ratio"] = np.where(df_bereinigt["technologie"] == "HIT", 0.82, np.random.uniform(0.70, 0.79, size=100))

print("-> Datenbereinigung und PR-Modellierung erfolgreich abgeschlossen.")
df_bereinigt.head()

--- FASE 1: ETL-Prozess (Bereinigung & Imputation mit Pandas) ---
-> Rohdaten erfolgreich geladen. Fehlwerte in 'leistung_kw': 7
-> Datenbereinigung und PR-Modellierung erfolgreich abgeschlossen.


,zeitstempel,technologie,einstrahlung_wm2,leistung_kw,performance_ratio
0,2026-06-01 00:00:00,CIGS,358.972545,5.746452,0.772910
1,2026-06-01 00:15:00,PERC,204.417694,31.581888,0.778037
2,2026-06-01 00:30:00,CIGS,852.369143,15.192310,0.782192
3,2026-06-01 00:45:00,CIGS,765.485875,34.032776,0.746021
4,2026-06-01 01:00:00,PERC,783.205734,12.846489,0.745136


### FASE 2: Relationales Datenbank-Schema (PostgreSQL)

Mithilfe von **pgAdmin 4** wurde die folgende relationale Datenbankstruktur in PostgreSQL erfolgreich initialisiert, um die bereinigten Zeitreihendaten effizient zu speichern:

```sql
-- Tabelle für die Solartechnologien (Standardisierung)
CREATE TABLE technologien (
    technologie_id SERIAL PRIMARY KEY,
    name VARCHAR(20) NOT NULL UNIQUE,
    beschreibung TEXT
);

-- Tabelle für historische Messdaten (Optimierte relationale Struktur)
CREATE TABLE messwerte_anlagen (
    messung_id SERIAL PRIMARY KEY,
    zeitstempel TIMESTAMP NOT NULL,
    technologie_id INT REFERENCES technologien(technologie_id),
    einstrahlung_wm2 NUMERIC(6,2),
    leistung_kw NUMERIC(5,2),
    performance_ratio NUMERIC(4,3)
);

In [2]:
import psycopg2
from psycopg2 import extras

print("--- FASE 3: Datenladevorgang (Jupyter -> PostgreSQL) ---")

try:
    # Verbindung zur lokalen PostgreSQL-Datenbank herstellen
    connection = psycopg2.connect(
        dbname="zee_solardaten_db",
        user="postgres",
        password="", 
        host="localhost",
        port="5432"
    )
    cursor = connection.cursor()
    print("-> Erfolgreich mit der Datenbank 'zee_solardaten_db' verbunden.")

    # Relationales Mapping für die IDs (PERC=1, HIT=2, CIGS=3)
    tech_mapping = {"PERC": 1, "HIT": 2, "CIGS": 3}

    # SQL-Query für den Massenimport (Bulk Insert)
    insert_query = """
        INSERT INTO messwerte_anlagen (zeitstempel, technologie_id, einstrahlung_wm2, leistung_kw, performance_ratio)
        VALUES (%s, %s, %s, %s, %s);
    """

    # Konvertierung des Pandas DataFrames in eine Liste von Tupeln
    daten_liste = [
        (
            row["zeitstempel"],
            tech_mapping[row["technologie"]],
            float(row["einstrahlung_wm2"]),
            float(row["leistung_kw"]),
            float(row["performance_ratio"])
        )
        for _, row in df_bereinigt.iterrows()
    ]

    # Effizientes Einfügen der Daten in PostgreSQL
    extras.execute_batch(cursor, insert_query, daten_liste)
    connection.commit()
    
    print(f"-> {len(daten_liste)} Datensätze erfolgreich in PostgreSQL eingefügt!")

except Exception as error:
    print(f"X Fehler beim Laden der Daten: {error}")

finally:
    if connection:
        cursor.close()
        connection.close()
        print("-> Datenbankverbindung geschlossen.")

--- FASE 3: Datenladevorgang (Jupyter -> PostgreSQL) ---
-> Erfolgreich mit der Datenbank 'zee_solardaten_db' verbunden.
-> 100 Datensätze erfolgreich in PostgreSQL eingefügt!
-> Datenbankverbindung geschlossen.


In [3]:
print("--- FASE 4: Validierung & KPI-Controlling aus SQL ---")

try:
    # 1. Erneute Verbindung für die Abfrage aufbauen
    connection = psycopg2.connect(
        dbname="zee_solardaten_db",
        user="postgres",
        password="",  
        host="localhost",
        port="5432"
    )
    
    # 2. SQL-Query mit einem JOIN, um die durchschnittliche Performance Ratio zu berechnen
    query = """
        SELECT t.name AS technologie, 
               ROUND(AVG(m.performance_ratio), 3) AS durchschnittliche_pr,
               ROUND(AVG(m.leistung_kw), 2) AS mittlere_leistung_kw
        FROM messwerte_anlagen m
        JOIN technologien t ON m.technologie_id = t.technologie_id
        GROUP BY t.name;
    """
    
    # 3. Daten direkt in ein Pandas DataFrame laden
    df_sql_kpis = pd.read_sql_query(query, connection)
    
    print("-> Analyse erfolgreich direkt aus der PostgreSQL-Datenbank geladen:")
    display(df_sql_kpis)

except Exception as error:
    print(f"X Fehler bei der Validierung: {error}")
    
finally:
    if connection:
        connection.close()
        print("-> Datenbankverbindung sicher geschlossen.")

--- FASE 4: Validierung & KPI-Controlling aus SQL ---
-> Analyse erfolgreich direkt aus der PostgreSQL-Datenbank geladen:


C:\Users\dgoo2\AppData\Local\Temp\ipykernel_15244\4282762656.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_kpis = pd.read_sql_query(query, connection)


,technologie,durchschnittliche_pr,mittlere_leistung_kw
0,CIGS,0.749,26.63
1,HIT,0.820,26.21
2,PERC,0.745,31.58


-> Datenbankverbindung sicher geschlossen.
